# Challenge Two — RAG in BigQuery
**Author:** Aaron

A retrieval-augmented chatbot for the fictional town of **Aurora Bay, Alaska**, built natively in
BigQuery: load the FAQ CSV, generate embeddings with `ML.GENERATE_EMBEDDING`, retrieve with
`VECTOR_SEARCH`, and pass the matches + question to Gemini for a grounded answer.

## Requirement -> implementation
| # | Requirement | Where |
|---|---|---|
| 1 | Load the Aurora Bay FAQ CSV into a BigQuery table | `load_table_from_uri` -> `faqs_raw` |
| 2 | Generate embeddings per Q&A record, store them in BigQuery | `ML.GENERATE_EMBEDDING` -> `faqs_embedded` |
| 3 | Chatbot that vector-searches the embeddings | `retrieve()` -> `VECTOR_SEARCH` |
| 4 | Pass retrieved data + question to Gemini | `answer()` (grounded Gen AI call) |

## Pipeline
```
CSV (GCS) -> faqs_raw -> ML.GENERATE_EMBEDDING -> faqs_embedded
question -> embed -> VECTOR_SEARCH(top_k) -> context + question -> Gemini -> grounded answer
```


## One-time setup (lab project already has most of this)
- **APIs:** `bigquery.googleapis.com`, `bigqueryconnection.googleapis.com`, `aiplatform.googleapis.com`.
- **IAM (you):** BigQuery Admin (`roles/bigquery.admin`) and, to grant the connection's service
  account, Project IAM Admin (`roles/resourcemanager.projectIamAdmin`).
- The notebook creates a **Cloud resource connection** and grants its service account
  `roles/aiplatform.user` so BigQuery ML can call the Vertex embedding model. If the lab already
  gave you a connection, set `CONNECTION_ID` to it and the create step becomes a no-op.

Colab Enterprise supplies Application Default Credentials — no keys.

## 1. Install & configure

In [13]:
%pip install --quiet --upgrade google-cloud-bigquery google-cloud-bigquery-connection google-genai

In [14]:
import os, time
import google.auth

try:
    _creds, _adc_project = google.auth.default()
except Exception:
    _adc_project = None
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or _adc_project
assert PROJECT_ID, "Could not determine the project. Set GOOGLE_CLOUD_PROJECT."

# BigQuery dataset/model/connection all live in this location (must match each other).
BQ_LOCATION    = "US"                      # multi-region; broadest BQML remote-model support
DATASET        = "aurora_bay"
CONNECTION_ID  = "embedding_conn"          # set to your lab's connection if one was pre-created
EMBEDDING_ENDPOINT = "text-embedding-005"  # supported BQML text-embedding endpoint
GENAI_LOCATION = "global"                  # Gemini endpoint for the final answer

SOURCE_CSV = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

RAW_TABLE      = f"{PROJECT_ID}.{DATASET}.faqs_raw"
EMB_TABLE      = f"{PROJECT_ID}.{DATASET}.faqs_embedded"
EMB_MODEL      = f"{PROJECT_ID}.{DATASET}.embedding_model"
CONNECTION_REF = f"{PROJECT_ID}.{BQ_LOCATION}.{CONNECTION_ID}"

print("Project:", PROJECT_ID)

Project: qwiklabs-gcp-02-d49a612b5dc6


In [15]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

def run_sql(sql, params=None):
    """Run a query (DDL or SELECT) and return the result rows."""
    job_config = bigquery.QueryJobConfig(query_parameters=params or [])
    return bq.query(sql, job_config=job_config).result()

# Create the dataset if it doesn't exist.
bq.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET}"), exists_ok=True)
print("Dataset ready:", DATASET)

Dataset ready: aurora_bay


## Requirement 1 — load the FAQ CSV into BigQuery

Autodetect the schema straight from the GCS CSV. After loading we inspect the columns so the rest
of the notebook adapts to whatever fields the file actually has.

In [16]:
load_cfg = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
bq.load_table_from_uri(SOURCE_CSV, RAW_TABLE, job_config=load_cfg).result()

tbl = bq.get_table(RAW_TABLE)
ORIG_COLS   = [f.name for f in tbl.schema]
STRING_COLS = [f.name for f in tbl.schema if f.field_type == "STRING"]
print(f"Loaded {tbl.num_rows} rows. Columns: {ORIG_COLS}")

# Peek at the data.
for row in run_sql(f"SELECT * FROM `{RAW_TABLE}` LIMIT 3"):
    print(dict(row))

Loaded 50 rows. Columns: ['string_field_0', 'string_field_1']
{'string_field_0': 'When was Aurora Bay founded?', 'string_field_1': 'Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.'}
{'string_field_0': 'What is the population of Aurora Bay?', 'string_field_1': 'Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.'}
{'string_field_0': 'Where is the Aurora Bay Town Hall located?', 'string_field_1': 'The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.'}


## Requirement 2 — generate embeddings, store them in BigQuery

**Connection + remote model.** BigQuery ML reaches the Vertex embedding model through a Cloud
resource connection whose service account has `roles/aiplatform.user`. This cell creates the
connection if needed, grants the role, and waits for IAM to propagate. *(If your lab pre-created a
connection, set `CONNECTION_ID` above and this still no-ops cleanly.)*

In [19]:
from google.cloud import bigquery_connection_v1 as bqc
from google.api_core.exceptions import AlreadyExists
import subprocess, time

conn_client = bqc.ConnectionServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{BQ_LOCATION}"
conn_name = f"{parent}/connections/{CONNECTION_ID}"

try:
    created = conn_client.create_connection(
        parent=parent, connection_id=CONNECTION_ID,
        connection=bqc.Connection(cloud_resource=bqc.CloudResourceProperties()),
    )
    conn_sa = created.cloud_resource.service_account_id
    print("Created connection:", CONNECTION_ID)
except AlreadyExists:
    conn_sa = conn_client.get_connection(name=conn_name).cloud_resource.service_account_id
    print("Connection exists:", CONNECTION_ID)

print("Connection service account:", conn_sa)

ROLE, MEMBER = "roles/aiplatform.user", f"serviceAccount:{conn_sa}"

# Grant the connection SA permission to call Vertex, then wait for propagation.
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=serviceAccount:{conn_sa} --role=roles/aiplatform.user \
    --condition=None --quiet >/dev/null 2>&1
print("Granted roles/aiplatform.user; waiting 30s for IAM to propagate...")
time.sleep(30)

def has_binding():
  out = subprocess.run(
      ["gcloud", "projects", "get-iam-policy", PROJECT_ID,
       "--flatten=bindings[].members",
       f"--filter=bindings.role={ROLE} AND bindings.members={MEMBER}",
       "--format=value(bindings.role)"],
      capture_output=True, text=True
  )

  return ROLE in out.stdout

for _ in range(10):
  if has_binding():
    print(f"Confirmed: {conn_sa} has {ROLE}")
    break
  print("  role not visible yet; sleeping 30s ... ")
  time.sleep(30)
else:
  raise RuntimeError(f"{ROLE} not granted to {conn_sa}")

Connection exists: embedding_conn
Connection service account: bqcx-244952544273-ujzn@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Granted roles/aiplatform.user; waiting 30s for IAM to propagate...
Confirmed: bqcx-244952544273-ujzn@gcp-sa-bigquery-condel.iam.gserviceaccount.com has roles/aiplatform.user


In [20]:
# Remote model over the Vertex text-embedding endpoint.
run_sql(f"""
CREATE OR REPLACE MODEL `{EMB_MODEL}`
REMOTE WITH CONNECTION `{CONNECTION_REF}`
OPTIONS (ENDPOINT = '{EMBEDDING_ENDPOINT}')
""")
print("Embedding model ready:", EMB_MODEL)

Embedding model ready: qwiklabs-gcp-02-d49a612b5dc6.aurora_bay.embedding_model


In [21]:
# Build the text to embed per row: label each string column so the Q&A pair is captured.
content_expr = "CONCAT(" + ", ".join(
    f"'{c}: ', IFNULL(CAST(`{c}` AS STRING), ''), '\\n'" for c in STRING_COLS
) + ")"
orig_select = ", ".join(f"`{c}`" for c in ORIG_COLS)

# Generate embeddings and store them alongside the original fields.
run_sql(f"""
CREATE OR REPLACE TABLE `{EMB_TABLE}` AS
SELECT {orig_select}, ml_generate_embedding_result AS embedding
FROM ML.GENERATE_EMBEDDING(
  MODEL `{EMB_MODEL}`,
  (SELECT *, {content_expr} AS content FROM `{RAW_TABLE}`),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type)
)
WHERE ml_generate_embedding_status = ''
""")

n = list(run_sql(f"SELECT COUNT(*) AS n, ARRAY_LENGTH(ANY_VALUE(embedding)) AS dims FROM `{EMB_TABLE}`"))[0]
print(f"Embedded {n['n']} rows, {n['dims']}-dim vectors stored in faqs_embedded.")

Embedded 50 rows, 768-dim vectors stored in faqs_embedded.


## Requirement 3 — retrieve with VECTOR_SEARCH

Embed the user's question (`RETRIEVAL_QUERY` task type) and find the nearest FAQ rows by cosine
distance. No vector index is created: at FAQ scale `VECTOR_SEARCH` does an exact brute-force scan,
and indexes require far more rows to be worthwhile.

In [ ]:
def retrieve(question, k=3):
    """Return the top-k most similar FAQ rows for a question."""
    sql = f"""
    SELECT base.*, distance
    FROM VECTOR_SEARCH(
      TABLE `{EMB_TABLE}`, 'embedding',
      (SELECT ml_generate_embedding_result AS embedding
       FROM ML.GENERATE_EMBEDDING(
         MODEL `{EMB_MODEL}`,
         (SELECT @q AS content),
         STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type))),
      top_k => @k, distance_type => 'COSINE')
    ORDER BY distance
    """
    params = [
        bigquery.ScalarQueryParameter("q", "STRING", question),
        bigquery.ScalarQueryParameter("k", "INT64", k),
    ]
    return [dict(r) for r in run_sql(sql, params)]

# Quick check.
for r in retrieve("Who is the mayor?", k=2):
    preview = {c: r[c] for c in STRING_COLS}
    print(round(r["distance"], 4), preview)

0.3624 {'string_field_0': 'Who is the current mayor of Aurora Bay?', 'string_field_1': 'The current mayor is Linda Greenwood, elected in 2021 for a four-year term.'}
0.4452 {'string_field_0': 'Where can I find official town announcements?', 'string_field_1': 'Official announcements are posted on the Aurora Bay official website, the Town Hall notice board, and broadcast on KABY-FM.'}


## Requirement 4 — pass retrieved data + question to Gemini

The retrieved rows become grounding context. The model is instructed to answer **only** from that
context and to say so when the answer isn't present — no making things up.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location=GENAI_LOCATION)

def resolve_model(candidates):
    for m in candidates:
        try:
            client.models.generate_content(
                model=m, contents="ping",
                config=types.GenerateContentConfig(max_output_tokens=8))
            return m
        except Exception as e:
            print(f"  {m} unavailable: {type(e).__name__}")
    raise RuntimeError("No candidate Gemini model available.")

MODEL = resolve_model(["gemini-3.1-flash", "gemini-2.5-flash", "gemini-2.0-flash"])
print("Using model:", MODEL)

SYSTEM_INSTRUCTIONS = """You are the Aurora Bay, Alaska town information assistant.
Answer the user's question using ONLY the provided FAQ context.
If the context does not contain the answer, say you don't have that information and suggest
contacting the town office. Be concise and accurate. Do not invent facts."""

def answer(question, k=3):
    rows = retrieve(question, k=k)
    context = "\n\n".join(
        "\n".join(f"{c}: {r[c]}" for c in STRING_COLS) for r in rows
    )
    prompt = f"FAQ CONTEXT:\n{context}\n\nUSER QUESTION: {question}"
    resp = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTIONS, temperature=0.2))
    return resp.text

## Demonstration (final step — leave the output saved)

> **Before exporting:** Runtime -> *Run all*, confirm the output below is populated, then
> File -> Download -> `.ipynb`.

In [ ]:
questions = [
    "Who is the current mayor of Aurora Bay?",            # in the FAQ
    "What are the operating hours of the public library?", # in the FAQ
    "What are the primary industries in Aurora Bay?",      # in the FAQ
    "What is the weather forecast for tomorrow?",          # NOT in the FAQ -> should decline
]

for q in questions:
    print("=" * 88)
    print("Q:", q)
    print("-" * 88)
    print(answer(q))
print("=" * 88)

## Submission notes
- If the embedding-model cell fails with a permissions error, the connection SA grant hasn't
  propagated yet — re-run after a minute, or confirm `conn_sa` has `roles/aiplatform.user`.
- If a column name from `ML.GENERATE_EMBEDDING` differs in your BQ version, run
  `SELECT * FROM ML.GENERATE_EMBEDDING(...) LIMIT 1` once to confirm and adjust.
- Run all top-to-bottom so the demo outputs are saved, then commit to `challenge2/` in the repo.
